
# Ensembl Liftoff vs Ssal_v3.1 native comparison

This notebook inspects the gene-level mapping quality between the Ensembl
ICSASG_v2 annotation lifted to the Ssal_v3.1 assembly and the native
Ssal_v3.1 Ensembl annotation. The analysis relies on the streamed
`within_assembly_compare.py` output produced in `experiments/comparison_runs/ens_lift_vs_native.tsv`.



## Setup

The code below uses only the Python standard library (plus IPython display
helpers) to keep dependencies minimal. SVG plots are generated manually and
saved under `experiments/comparison_runs/figures` so they can be reused in
other documents.


In [ ]:

    from pathlib import Path
    import csv
    from collections import Counter
    import math
    from typing import Dict, Iterable, List, Tuple
    from IPython.display import SVG, display

    DATA_PATH = Path('..') / 'experiments' / 'comparison_runs' / 'ens_lift_vs_native.tsv'
    FIG_DIR = Path('..') / 'experiments' / 'comparison_runs' / 'figures'
    FIG_DIR.mkdir(parents=True, exist_ok=True)

    def parse_stats_field(raw: str) -> Dict[str, object]:
        result: Dict[str, object] = {}
        for chunk in raw.split(';'):
            if not chunk or '=' not in chunk:
                continue
            key, value = chunk.split('=', 1)
            value = value.strip()
            if not value:
                result[key] = value
                continue
            lower = value.lower()
            try:
                if lower == 'nan':
                    result[key] = float('nan')
                elif any(sep in value for sep in ('.', 'e', 'E')):
                    result[key] = float(value)
                else:
                    result[key] = int(value)
            except ValueError:
                result[key] = value
        return result

    def load_gene_records(path: Path) -> List[Dict[str, object]]:
        records: List[Dict[str, object]] = []
        with path.open('r', encoding='utf-8') as handle:
            reader = csv.DictReader(handle, delimiter='	')
            for row in reader:
                if row['feature'] != 'gene':
                    continue
                stats = parse_stats_field(row['stats'])
                record = {
                    'annA': row['annA'],
                    'geneA': row['geneA'],
                    'geneB': row['geneB'],
                    'stats': stats,
                    'class': stats.get('class', 'Unknown'),
                }
                records.append(record)
        return records

    def render_bar_chart(data_items: List[Tuple[str, int]], title: str, output_path: Path) -> None:
        if not data_items:
            output_path.write_text('<svg xmlns="http://www.w3.org/2000/svg" width="400" height="200"></svg>', encoding='utf-8')
            return
        width, height = 800, 420
        margin = 60
        max_value = max(value for _label, value in data_items) or 1
        bar_width = (width - 2 * margin) / len(data_items)
        scale = (height - 2 * margin) / max_value
        lines = [
            f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
            f'<title>{title}</title>',
            '<rect width="100%" height="100%" fill="#ffffff"/>',
            f'<text x="{width/2:.1f}" y="{margin/2:.1f}" text-anchor="middle" font-family="Helvetica" font-size="18">{title}</text>',
            f'<line x1="{margin}" y1="{height-margin}" x2="{width-margin}" y2="{height-margin}" stroke="#333"/>',
            f'<line x1="{margin}" y1="{margin}" x2="{margin}" y2="{height-margin}" stroke="#333"/>'
        ]
        for idx, (label, value) in enumerate(data_items):
            x = margin + idx * bar_width + bar_width * 0.1
            bar_h = value * scale
            y = height - margin - bar_h
            lines.append(f'<rect x="{x:.2f}" y="{y:.2f}" width="{bar_width*0.8:.2f}" height="{bar_h:.2f}" fill="#4477aa"/>')
            lines.append(f'<text x="{x + bar_width*0.4:.2f}" y="{height - margin + 20:.2f}" text-anchor="middle" font-family="Helvetica" font-size="14">{label}</text>')
            lines.append(f'<text x="{x + bar_width*0.4:.2f}" y="{y - 5:.2f}" text-anchor="middle" font-family="Helvetica" font-size="13">{value}</text>')
        for frac in (0.25, 0.5, 0.75, 1.0):
            y = height - margin - frac * max_value * scale
            lines.append(f'<line x1="{margin-5}" y1="{y:.2f}" x2="{width-margin}" y2="{y:.2f}" stroke="#cccccc" stroke-dasharray="4,4"/>')
            lines.append(f'<text x="{margin-10}" y="{y+4:.2f}" text-anchor="end" font-family="Helvetica" font-size="12">{int(frac*max_value)}</text>')
        lines.append('</svg>')
        output_path.write_text('
'.join(lines), encoding='utf-8')

    def render_histogram(values: List[float], bins: int, title: str, output_path: Path) -> None:
        clean = [v for v in values if isinstance(v, (int, float)) and not math.isnan(v)]
        if not clean:
            output_path.write_text('<svg xmlns="http://www.w3.org/2000/svg" width="400" height="200"></svg>', encoding='utf-8')
            return
        counts = [0] * bins
        for value in clean:
            clipped = min(max(value, 0.0), 0.999999)
            idx = int(clipped * bins)
            counts[idx] += 1
        width, height = 800, 420
        margin = 60
        max_value = max(counts) or 1
        bar_width = (width - 2 * margin) / bins
        scale = (height - 2 * margin) / max_value
        lines = [
            f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
            f'<title>{title}</title>',
            '<rect width="100%" height="100%" fill="#ffffff"/>',
            f'<text x="{width/2:.1f}" y="{margin/2:.1f}" text-anchor="middle" font-family="Helvetica" font-size="18">{title}</text>',
            f'<line x1="{margin}" y1="{height-margin}" x2="{width-margin}" y2="{height-margin}" stroke="#333"/>',
            f'<line x1="{margin}" y1="{margin}" x2="{margin}" y2="{height-margin}" stroke="#333"/>'
        ]
        for idx, count in enumerate(counts):
            x = margin + idx * bar_width
            bar_h = count * scale
            y = height - margin - bar_h
            lines.append(f'<rect x="{x:.2f}" y="{y:.2f}" width="{bar_width - 1:.2f}" height="{bar_h:.2f}" fill="#66aa55"/>')
        for tick in [0.0, 0.25, 0.5, 0.75, 1.0]:
            x = margin + tick * (width - 2 * margin)
            lines.append(f'<line x1="{x:.2f}" y1="{height-margin}" x2="{x:.2f}" y2="{height-margin+5}" stroke="#333"/>')
            lines.append(f'<text x="{x:.2f}" y="{height-margin+25:.2f}" text-anchor="middle" font-family="Helvetica" font-size="12">{tick:.2f}</text>')
        for frac in (0.25, 0.5, 0.75, 1.0):
            y = height - margin - frac * max_value * scale
            lines.append(f'<line x1="{margin-5}" y1="{y:.2f}" x2="{width-margin}" y2="{y:.2f}" stroke="#cccccc" stroke-dasharray="4,4"/>')
            lines.append(f'<text x="{margin-10}" y="{y+4:.2f}" text-anchor="end" font-family="Helvetica" font-size="12">{int(frac*max_value)}</text>')
        lines.append('</svg>')
        output_path.write_text('
'.join(lines), encoding='utf-8')

    def extract_stable_id(gene_id: str) -> str:
        return gene_id.split(':', 1)[-1]



## Load comparison output


In [ ]:

records = load_gene_records(DATA_PATH)
total_pairs = len(records)
unique_genes_a = {r['geneA'] for r in records}
unique_genes_b = {r['geneB'] for r in records}
print(f"Gene pairs analysed: {total_pairs}")
print(f"Unique lifted genes with overlaps: {len(unique_genes_a)}")
print(f"Unique native genes with overlaps: {len(unique_genes_b)}")
records[:2]



## Mapping quality summary


In [ ]:

    class_order = ['Green', 'Yellow', 'Red', 'NotMapped']
    class_counts = Counter(r['class'] for r in records)
    ordered_counts = [(label, class_counts.get(label, 0)) for label in class_order if class_counts.get(label, 0) > 0]
    for label, value in ordered_counts:
        print(f"{label}: {value}")
    not_mapped = class_counts.get('NotMapped', 0)
    print(f"
Total NotMapped pairs: {not_mapped}")
    class_chart_path = FIG_DIR / 'class_counts.svg'
    render_bar_chart(ordered_counts, 'Gene-level classification counts', class_chart_path)
    display(SVG(filename=str(class_chart_path)))



## Stable Ensembl gene IDs


In [ ]:

    stable_counter = Counter()
    per_class_match = Counter()
    for record in records:
        stable_a = extract_stable_id(record['geneA'])
        stable_b = extract_stable_id(record['geneB'])
        if stable_a == stable_b:
            stable_counter['same_id'] += 1
            per_class_match[record['class'], 'same'] += 1
        else:
            stable_counter['different_id'] += 1
            per_class_match[record['class'], 'different'] += 1
    total = stable_counter['same_id'] + stable_counter['different_id']
    print(f"Pairs with identical stable IDs: {stable_counter['same_id']} ({stable_counter['same_id'] / total:.1%})")
    print(f"Pairs with different stable IDs: {stable_counter['different_id']} ({stable_counter['different_id'] / total:.1%})")
    print('
Breakdown by class:')
    for label in class_order:
        same = per_class_match.get((label, 'same'), 0)
        diff = per_class_match.get((label, 'different'), 0)
        total_cls = same + diff
        if total_cls == 0:
            continue
        print(f"  {label}: {same} same ({same/total_cls:.1%}), {diff} different ({diff/total_cls:.1%})")



## Distribution of overlap metrics


In [ ]:

def to_float_list(values):
    result: List[float] = []
    for value in values:
        if isinstance(value, (int, float)):
            result.append(float(value))
        elif isinstance(value, str):
            try:
                result.append(float(value))
            except ValueError:
                result.append(float('nan'))
        else:
            result.append(float('nan'))
    return result

cds_values = to_float_list([record['stats'].get('best_jaccard_cds_phase') for record in records])
exon_values = to_float_list([record['stats'].get('best_jaccard_exon') for record in records])
cds_chart_path = FIG_DIR / 'best_jaccard_cds_hist.svg'
exon_chart_path = FIG_DIR / 'best_jaccard_exon_hist.svg'
render_histogram(cds_values, 20, 'Best CDS-phase Jaccard', cds_chart_path)
render_histogram(exon_values, 20, 'Best exon Jaccard', exon_chart_path)
display(SVG(filename=str(cds_chart_path)))
display(SVG(filename=str(exon_chart_path)))



## Notes

* `NotMapped` pairs arise when the lifted transcripts overlap in space but do
  not meet the minimum evidence thresholds or represent antisense conflicts.
* Additional downgrade rules (competing partners, frame inconsistencies, etc.)
  will refine the classification once implemented in the comparison script.
